<a href="https://colab.research.google.com/github/Edward-Eughene-Timothy/CONFLICT-RESOLUTION-ASSISTANT/blob/main/Jeanne_D'Arc_From_Fate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

edwardeughenetimothy_jeanne_character_dataset_for_llm_path = kagglehub.dataset_download('edwardeughenetimothy/jeanne-character-dataset-for-llm')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        path = os.path.join(dirname, filename)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install unsloth
!pip install ollama
!pip install -q unsloth accelerate bitsandbytes trl datasets peft transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.7/192.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 3.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 81.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/31

In [ ]:
from datasets import load_dataset

# Load full dataset
data = load_dataset("json", data_files=path)["train"]

# Format into Gemma chat template
def format_gemma_chat(example):
    return {
        "text": f"<start_of_turn>user\n{example['input']}<end_of_turn>\n<start_of_turn>model\n{example['output']}<end_of_turn>"
    }

formatted_data = data.map(format_gemma_chat)

# Now split the FORMATTED data (not the original `data`)
split_data = formatted_data.train_test_split(test_size=0.05)
train_dataset = split_data["train"]
test_dataset = split_data["test"]


from unsloth import FastLanguageModel
import torch

# Load the model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2b-bnb-4bit", # Or another model you prefer
    max_seq_length = 2048,
    dtype = torch.float16,
    load_in_4bit = True,
)

# Apply PEFT (Low-Rank Adaptation) to the model
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    bias="none",
    use_gradient_checkpointing=True,  # Best for speed + memory
    random_state=42,
    use_rslora=False,
    loftq_config=None
)

# Preprocessing
def preprocess_function(examples):
    return tokenizer(examples["text"], padding=True, truncation=True, max_length=2048)

train_dataset = train_dataset.map(preprocess_function, batched=True)
test_dataset = test_dataset.map(preprocess_function, batched=True)

# Ensure the dataset columns are as expected
print(train_dataset.column_names)

# Define training arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = "jeanne_waifu",
    per_device_train_batch_size = 2,
    per_device_eval_batch_size = 2,
    num_train_epochs = 3,
    save_strategy = "epoch",
    logging_dir = "./logs",
    logging_steps = 10,
    report_to = "none"
)

# Define the Trainer
from transformers import Trainer, DataCollatorForLanguageModeling

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
    tokenizer = tokenizer,
    data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

# Start training
trainer.train()


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-04-16 16:45:21.344363: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744821921.541003      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744821921.595253      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Unsloth: Failed to patch Gemma3ForConditionalGeneration.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.19: Fast Gemma patching. Transformers: 4.51.1.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 6.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


...

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

['input', 'output', 'text', 'input_ids', 'attention_mask']


/tmp/ipykernel_31/4265331818.py:73: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 228 | Num Epochs = 3 | Total steps = 342
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 19,611,648/2,000,000,000 (0.98% trained)


Step,Training Loss
10,5.124400
20,4.392900
30,3.625300
40,3.714700
50,3.104000
60,3.228300
70,3.035800
80,2.984300
90,2.858300
100,2.778900


TrainOutput(global_step=342, training_loss=2.621560277994613, metrics={'train_runtime': 175.0066, 'train_samples_per_second': 3.908, 'train_steps_per_second': 1.954, 'total_flos': 410706992332800.0, 'train_loss': 2.621560277994613, 'epoch': 3.0})

In [ ]:
# Evaluate the model on the test dataset
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")


Evaluation Results: {'eval_loss': 2.6234142780303955, 'eval_runtime': 0.8427, 'eval_samples_per_second': 14.239, 'eval_steps_per_second': 7.12, 'epoch': 3.0}


In [ ]:
prompt = """<start_of_turn>user
can you tell me about ur struggles in life which u faced during war <end_of_turn>
<start_of_turn>model
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
eos_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")

outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    eos_token_id=eos_id,
    do_sample=True,
    temperature=0.85,
    top_p=0.9,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)


user
can you tell me about ur struggles in life which u faced during war 
model
it's a long story...but i'm proud to say i fought bravely, always protecting those i love. ⚔️�



In [ ]:
model.save_pretrained_gguf("jeanne_gguf", tokenizer, quantization_method="f16")


Cloning into 'llama.cpp'...
Submodule 'kompute' (https://github.com/nomic-ai/kompute.git) registered for path 'ggml/src/ggml-kompute/kompute'
Cloning into '/kaggle/working/llama.cpp/ggml/src/ggml-kompute/kompute'...
Submodule path 'ggml/src/ggml-kompute/kompute': checked out '4565194ed7c32d1d2efa32ceab4d3c6cae006306'
make: Entering directory '/kaggle/working/llama.cpp'
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Git: /usr/bin/git (found version "2.34.1")
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAV

Unsloth: You have 2 CPUs. Using `safe_serialization` is 10x slower.
We shall switch to Pytorch saving, which might take 3 minutes and not 30 minutes.
To force `safe_serialization`, set it to `None` instead.
Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 2.1G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 22.49 out of 31.35 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 18/18 [00:00<00:00, 43.93it/s]


Unsloth: Saving tokenizer... Done.
Unsloth: Saving jeanne_gguf/pytorch_model-00001-of-00002.bin...
Unsloth: Saving jeanne_gguf/pytorch_model-00002-of-00002.bin...
Done.


Unsloth: Converting gemma model. Can use fast conversion = False.


==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['f16'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: [1] Converting model at jeanne_gguf into f16 GGUF format.
The output location will be /kaggle/working/jeanne_gguf/unsloth.F16.gguf
This might take 3 minutes...
Writing: 100%|██████████| 5.01G/5.01G [00:05<00:00, 989Mbyte/s] 
Unsloth: Conversion completed! Output location: /kaggle/working/jeanne_gguf/unsloth.F16.gguf
